# C++ vs agent-translated Python parity

Build and run the original C++ core pipeline, run the agent-translated Python port on the same Trial32 inputs, then compare catalogs.

The full C++ file depends on an in-house math/stats library that is not in this repo. `legacy/cpp_core/build_core.py` extracts the **core** sections (parse → detect → write markup / subavalanche / avalanche / structure), links against local stubs, and produces `legacy/cpp_core/build/lmp_swapstat4_core`.

In [23]:
from __future__ import annotations

import importlib.util
import pathlib
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "pyproject.toml").is_file() and (base / "legacy").is_dir():
            return base
    raise FileNotFoundError(f"Could not locate the repository above {here}")


ROOT = find_repo_root()
SWAP = ROOT / (
    "data/raw/trial32/"
    "N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt"
)
CONFIG = ROOT / "configs/trial32/swapstat.params.ini"
STEM = SWAP.name[:-4]
OUT_PREFIX = "trial32_dt10-dr400"

CPP_BUILD_SCRIPT = ROOT / "legacy/cpp_core/build_core.py"
CPP_BIN = ROOT / "legacy/cpp_core/build/lmp_swapstat4_core"
CPP_DIR = ROOT / "outputs/trial32/cpp_core"
PY_DIR = ROOT / "outputs/trial32/python_port"

MARKUP_COLS = [
    "time",
    "swap_num",
    "x",
    "y",
    "z",
    "surface_area",
    "subavalanche_id",
    "subavalanche_size",
    "avalanche_id",
    "avalanche_size",
]
SUBAV_RENAME = {
    "Subavalanche #": "event_id",
    "Number of swaps": "n_swaps",
    "Subavalanche P50 time": "time_p50",
    "Subavalanche start time": "time_start",
    "Subavalanche end time": "time_end",
    "Subavalanche duration": "duration",
    "Subavalanche diameter": "diameter",
    "Subavalanche Rg": "radius_of_gyration",
    "Subavalanche I1": "inertia_1",
    "Subavalanche I2": "inertia_2",
    "Subavalanche X position": "x",
    "Subavalanche Y position": "y",
    "Simultaneous subavalanches count": "simultaneous_count",
}
AV_RENAME = {
    "Avalanche ID": "event_id",
    "Number of subavalanches": "n_subavalanches",
    "Number of swaps": "n_swaps",
    "Number of swaps (direction corrected)": "direction_corrected_swaps",
    "Avalanche start time": "time_start",
    "Avalanche end time": "time_end",
    "Avalanche avg time": "time_average",
    "Avalanche P50 time": "time_p50",
    "Avalanche duration": "duration",
    "Avalanche Tg": "temporal_gyration",
    "Avalanche Rg": "radius_of_gyration",
    "Avalanche Diameter": "diameter",
}


## Shared run helper

Both binaries write markup / subav / av files next to the swap path. Use an isolated output directory with a symlink to the immutable raw swap file.

In [24]:
def prepare_run_dir(directory: pathlib.Path) -> tuple[pathlib.Path, pathlib.Path]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)
    swap_link = directory / SWAP.name
    swap_link.symlink_to(SWAP.resolve())
    names = directory / "names.txt"
    names.write_text(f"label 1\n{swap_link} 1e9\n")
    out_prefix = directory / OUT_PREFIX
    return names, out_prefix


## Build and run the C++ core binary

In [25]:
print("Building C++ core...")
subprocess.run([sys.executable, str(CPP_BUILD_SCRIPT)], check=True)
assert CPP_BIN.is_file(), CPP_BIN

names, out_prefix = prepare_run_dir(CPP_DIR)
print(f"Running {CPP_BIN.name} → {CPP_DIR}")
subprocess.run(
    [str(CPP_BIN), str(names), str(CONFIG), str(out_prefix)],
    check=True,
)
print("C++ outputs:", sorted(p.name for p in CPP_DIR.glob("*.txt")))


Building C++ core...
wrote /home/neriberman/REPOS/crumpling/legacy/cpp_core/build/lmp_swapstat4_main_core.cpp (2133 lines)
compile: g++ -std=c++17 -O2 -Wall -Wno-unused-variable -Wno-unused-but-set-variable -Wno-sign-compare -I/home/neriberman/REPOS/crumpling/legacy/cpp_core/vendor /home/neriberman/REPOS/crumpling/legacy/cpp_core/build/lmp_swapstat4_main_core.cpp -o /home/neriberman/REPOS/crumpling/legacy/cpp_core/build/lmp_swapstat4_core -lm
binary: /home/neriberman/REPOS/crumpling/legacy/cpp_core/build/lmp_swapstat4_core
Running lmp_swapstat4_core → /home/neriberman/REPOS/crumpling/outputs/trial32/cpp_core

Read ini file...done.
done.
Read swap files...
File /home/neriberman/REPOS/crumpling/outputs/trial32/cpp_core/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt
Records read: 3242, errors 0
done.
Detect subavalanches...
done.
Detect avalanches...
avalancheDetectionMechanism == 0
done.

[core build] skipping histogram/statistics sections...

Writing swap d

## Run the translated Python port

In [26]:
def load_legacy_main():
    path = ROOT / "legacy/lmp_swapstat4_main.py"
    spec = importlib.util.spec_from_file_location("lmp_swapstat4_main", path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


names, out_prefix = prepare_run_dir(PY_DIR)
module = load_legacy_main()
rc = module.main([str(names), str(CONFIG), str(out_prefix)])
if rc != 0:
    raise RuntimeError(f"lmp_swapstat4_main.main returned {rc}")
print(f"Python port outputs → {PY_DIR}")
print("files:", sorted(p.name for p in PY_DIR.glob("*.txt")))



Read ini file...done.
done.
Read swap files...
/home/neriberman/REPOS/crumpling/outputs/trial32/python_port/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt
Records read: 3242, errors 0
done.
Detect subavalanches...
done.
Detect avalanches...
avalancheDetectionMechanism == 0
done.

Writing swap data... done.

Writing subavalanche data... done.

Writing avalanche data... done.

Writing all avalanche data... done.

Writing avalanche structure data... done.
Python port outputs → /home/neriberman/REPOS/crumpling/outputs/trial32/python_port
files: ['N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt', 'N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps__avdata.txt', 'N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps__markup.txt', 'N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps__subavdata.txt', 'names.txt', 'trial32_dt10-dr400__avdata.txt', 'trial32_dt10-dr400_avstruc

## Loaders

The markup header contains a malformed double-tab (empty column) that is **not** present in data rows. Loaders assign column names explicitly so both sides align.

In [27]:
def load_markup(path: pathlib.Path) -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep="\t",
        header=0,
        names=MARKUP_COLS,
        usecols=range(len(MARKUP_COLS)),
    )


def load_subavalanches(path: pathlib.Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t")
    return frame.rename(columns=SUBAV_RENAME)[list(SUBAV_RENAME.values())]


def load_avalanches(path: pathlib.Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t").dropna(how="all")
    return frame.rename(columns=AV_RENAME)[list(AV_RENAME.values())]


def catalog_paths(directory: pathlib.Path) -> dict[str, pathlib.Path]:
    return {
        "markup": directory / f"{STEM}__markup.txt",
        "subavalanches": directory / f"{STEM}__subavdata.txt",
        "avalanches": directory / f"{STEM}__avdata.txt",
        "av_all": directory / f"{OUT_PREFIX}__avdata.txt",
        "av_structure": directory / f"{OUT_PREFIX}_avstructure.txt",
    }


cpp_paths = catalog_paths(CPP_DIR)
py_paths = catalog_paths(PY_DIR)

cpp = {
    "markup": load_markup(cpp_paths["markup"]),
    "subavalanches": load_subavalanches(cpp_paths["subavalanches"]),
    "avalanches": load_avalanches(cpp_paths["avalanches"]),
}
py = {
    "markup": load_markup(py_paths["markup"]),
    "subavalanches": load_subavalanches(py_paths["subavalanches"]),
    "avalanches": load_avalanches(py_paths["avalanches"]),
}

pd.DataFrame(
    {
        "catalog": list(cpp),
        "cpp_rows": [len(cpp[k]) for k in cpp],
        "py_rows": [len(py[k]) for k in cpp],
    }
)


,catalog,cpp_rows,py_rows
0,markup,3242,3242
1,subavalanches,892,892
2,avalanches,225,225


## Column-wise comparison

`left` = live C++ core run, `right` = live Python port.

Summary columns: `n_diff` (nonzero |cpp−py| count), `min`/`max`/`avg` of absolute and signed diffs over those rows, plus `within_tol` / `bit_exact`.


In [28]:
def compare_frames(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    name: str,
    rtol: float = 0.0,
    atol: float = 0.0,
) -> pd.DataFrame:
    if list(left.columns) != list(right.columns):
        raise ValueError(f"{name}: column mismatch {list(left.columns)} vs {list(right.columns)}")
    if len(left) != len(right):
        raise ValueError(f"{name}: row count {len(left)} vs {len(right)}")

    rows = []
    mismatch_mask = np.zeros(len(left), dtype=bool)
    for column in left.columns:
        a = left[column].to_numpy()
        b = right[column].to_numpy()
        if pd.api.types.is_numeric_dtype(left[column]):
            a_f = a.astype(float)
            b_f = b.astype(float)
            signed = a_f - b_f
            abs_diff = np.where(np.isnan(a_f) & np.isnan(b_f), 0.0, np.abs(signed))
            nonzero = abs_diff > 0
            n_diff = int(np.count_nonzero(nonzero))
            close = np.isclose(a_f, b_f, rtol=rtol, atol=atol, equal_nan=True)
            if n_diff:
                diffs = abs_diff[nonzero]
                signed_nz = signed[nonzero]
                min_abs = float(np.min(diffs))
                max_abs = float(np.max(diffs))
                avg_abs = float(np.mean(diffs))
                min_signed = float(np.min(signed_nz))
                max_signed = float(np.max(signed_nz))
                avg_signed = float(np.mean(signed_nz))
            else:
                min_abs = max_abs = avg_abs = 0.0
                min_signed = max_signed = avg_signed = 0.0
            rows.append(
                {
                    "catalog": name,
                    "column": column,
                    "n_diff": n_diff,
                    "min_abs_diff": min_abs,
                    "max_abs_diff": max_abs,
                    "avg_abs_diff": avg_abs,
                    "min_diff": min_signed,
                    "max_diff": max_signed,
                    "avg_diff": avg_signed,
                    "within_tol": bool(np.all(close)),
                    "bit_exact": bool(n_diff == 0),
                }
            )
            mismatch_mask |= ~close
        else:
            equal = a == b
            n_diff = int(np.size(equal) - np.count_nonzero(equal))
            rows.append(
                {
                    "catalog": name,
                    "column": column,
                    "n_diff": n_diff,
                    "min_abs_diff": np.nan,
                    "max_abs_diff": np.nan,
                    "avg_abs_diff": np.nan,
                    "min_diff": np.nan,
                    "max_diff": np.nan,
                    "avg_diff": np.nan,
                    "within_tol": bool(n_diff == 0),
                    "bit_exact": bool(n_diff == 0),
                }
            )
            mismatch_mask |= ~equal

    summary = pd.DataFrame(rows)
    n_bad = int(mismatch_mask.sum())
    print(
        f"{name}: {n_bad} mismatched rows of {len(left)}; "
        f"within_tol = {bool(summary['within_tol'].all())}; "
        f"bit_exact = {bool(summary['bit_exact'].all())}"
    )
    if n_bad:
        display_cols = list(left.columns)[:8]
        bad = pd.concat(
            [
                left.loc[mismatch_mask, display_cols].add_prefix("cpp_"),
                right.loc[mismatch_mask, display_cols].add_prefix("py_"),
            ],
            axis=1,
        )
        display(bad.head(10))
    return summary


# Geometry / time fields are printed with 3–4 decimals; C++ vs Python
# half-tie rounding can differ by one printed ULP (0.001 / 0.0001).
summaries = [
    compare_frames(cpp["markup"], py["markup"], name="markup"),
    compare_frames(
        cpp["subavalanches"], py["subavalanches"], name="subavalanches", atol=1.1e-3
    ),
    compare_frames(
        cpp["avalanches"], py["avalanches"], name="avalanches", atol=1.1e-4
    ),
]
summary = pd.concat(summaries, ignore_index=True)
summary


markup: 0 mismatched rows of 3242; within_tol = True; bit_exact = True
subavalanches: 0 mismatched rows of 892; within_tol = True; bit_exact = False
avalanches: 0 mismatched rows of 225; within_tol = True; bit_exact = False


,catalog,column,n_diff,min_abs_diff,max_abs_diff,avg_abs_diff,min_diff,max_diff,avg_diff,within_tol,bit_exact
0,markup,time,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
1,markup,swap_num,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
2,markup,x,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
3,markup,y,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
4,markup,z,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
5,markup,surface_area,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
6,markup,subavalanche_id,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
7,markup,subavalanche_size,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
8,markup,avalanche_id,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True
9,markup,avalanche_size,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,True,True


## Combined avalanche file and structure sanity checks

In [29]:
cpp_av_all = load_avalanches(cpp_paths["av_all"])
py_av_all = load_avalanches(py_paths["av_all"])
av_all_summary = compare_frames(cpp_av_all, py_av_all, name="av_all")

structure_match = (
    cpp_paths["av_structure"].read_bytes() == py_paths["av_structure"].read_bytes()
)
print("avalanche structure byte-identical:", structure_match)
pd.DataFrame(
    [
        {
            "side": "cpp",
            "n_bytes": cpp_paths["av_structure"].stat().st_size,
            "n_lines": len(cpp_paths["av_structure"].read_text().splitlines()),
        },
        {
            "side": "python",
            "n_bytes": py_paths["av_structure"].stat().st_size,
            "n_lines": len(py_paths["av_structure"].read_text().splitlines()),
        },
    ]
)


av_all: 2 mismatched rows of 225; within_tol = False; bit_exact = False


,cpp_event_id,cpp_n_subavalanches,cpp_n_swaps,cpp_direction_corrected_swaps,cpp_time_start,cpp_time_end,cpp_time_average,cpp_time_p50,py_event_id,py_n_subavalanches,py_n_swaps,py_direction_corrected_swaps,py_time_start,py_time_end,py_time_average,py_time_p50
9,10,1,4,4,46.803,60.615,52.6703,51.6315,10,1,4,4,46.803,60.615,52.6702,51.6315
159,160,6,20,20,21037.122,21208.356,21103.1480,21066.8955,160,6,20,20,21037.122,21208.356,21103.1479,21066.8955


avalanche structure byte-identical: True


,side,n_bytes,n_lines
0,cpp,17128166,70647
1,python,17128166,70647


## Verdict

In [30]:
full = pd.concat([summary, av_all_summary], ignore_index=True)
n_bad_cols = int((~full["within_tol"]).sum())
verdict = {
    "cpp_binary": str(CPP_BIN),
    "cpp_outdir": str(CPP_DIR),
    "py_outdir": str(PY_DIR),
    "markup_ok": bool(full.loc[full["catalog"] == "markup", "within_tol"].all()),
    "subavalanches_ok": bool(full.loc[full["catalog"] == "subavalanches", "within_tol"].all()),
    "avalanches_ok": bool(full.loc[full["catalog"] == "avalanches", "within_tol"].all()),
    "av_all_ok": bool(full.loc[full["catalog"] == "av_all", "within_tol"].all()),
    "av_structure_byte_identical": structure_match,
    "n_columns_outside_tol": n_bad_cols,
    "max_abs_diff_any_column": float(full["max_abs_diff"].max(skipna=True)),
}
display(pd.Series(verdict, name="value").to_frame())

if n_bad_cols == 0 and structure_match:
    print("PASS: live C++ core run matches the agent-translated Python port.")
else:
    print("FAIL: differences detected; inspect the summary tables above.")


,value
cpp_binary,/home/neriberman/REPOS/crumpling/legacy/cpp_co...
cpp_outdir,/home/neriberman/REPOS/crumpling/outputs/trial...
py_outdir,/home/neriberman/REPOS/crumpling/outputs/trial...
markup_ok,True
subavalanches_ok,True
avalanches_ok,True
av_all_ok,False
av_structure_byte_identical,True
n_columns_outside_tol,1
max_abs_diff_any_column,0.001


FAIL: differences detected; inspect the summary tables above.
